# Stacking & Voting Classifiers

#### Hard vs. Soft Voting
##### Dataset: from sklearn.datasets import fetch_california_housing
##### Objective: Learn to combine heterogenous models.
##### Task: Implement a VotingClassifier using scikit-learn.
##### Requirements:
              - Select three fundamentally different algorithms (e.g., Logistic Regression, Support Vector Machine, and Random Forest).              - Evaluate the ensemble using Hard Voting (majority rule).
              - Evaluate the ensemble using Soft Voting (averaging predicted probabilities). Note: Ensure the SVM is configured to output probabilities.
              - Compare the ensemble's performance against the best individual base model.


In [18]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

X, y = fetch_california_housing(return_X_y=True)
y = (y > np.median(y)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
svm = make_pipeline(StandardScaler(), SVC(probability=True))
rf = RandomForestClassifier(random_state=42)

models = [("LR", lr), ("SVM", svm), ("RF", rf)]

for name, model in models:
    model.fit(X_train, y_train)
    print(name, accuracy_score(y_test, model.predict(X_test)))

hard = VotingClassifier(models, voting="hard")
soft = VotingClassifier(models, voting="soft")

hard.fit(X_train, y_train)
soft.fit(X_train, y_train)

print("Hard Voting:", accuracy_score(y_test, hard.predict(X_test)))
print("Soft Voting:", accuracy_score(y_test, soft.predict(X_test)))

LR 0.8260658914728682
SVM 0.8602228682170543
RF 0.8934108527131783
Hard Voting: 0.8662790697674418
Soft Voting: 0.8737887596899225


Hard Voting chooses the class with the most votes.
Soft Voting uses average predicted probabilities.
SVM needs probability=True for Soft Voting.

#### The Sklearn Stacking Framework
##### Dataset: from sklearn.datasets import fetch_california_housing
##### Objective: Understand how meta-learners operate on the predictions of base learners.
##### Task: Build a two-level StackingClassifier or StackingRegressor.
##### Requirements:
              - Level-0 (Base Models): Use a mix of models (e.g., Ridge Regression, Random Forest, K-Nearest Neighbors).              
              - Level-1 (Meta-Model): Use a simple, interpretable model (e.g., Linear/Logistic Regression) to learn the weights of the level-0 models.
              - Inspect the coefficients of the Level-1 Linear Regression model. Which Level-0 model was trusted the most by the meta-learner?

In [19]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

base_models = [
    ("ridge", make_pipeline(StandardScaler(), Ridge())),
    ("random_forest", RandomForestRegressor(n_estimators=50, random_state=42)),
    ("knn", make_pipeline(StandardScaler(), KNeighborsRegressor()))
]

stack = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression()
)

stack.fit(X_train, y_train)

y_pred = stack.predict(X_test)
print("Stacking R2 Score:", r2_score(y_test, y_pred))

coefs = stack.final_estimator_.coef_

for name, coef in zip(["ridge", "random_forest", "knn"], coefs):
    print(name, "coefficient:", coef)

most_trusted = ["ridge", "random_forest", "knn"][np.argmax(np.abs(coefs))]
print("Most trusted model:", most_trusted)

Stacking R2 Score: 0.8046327751150714
ridge coefficient: -0.01204628635650771
random_forest coefficient: 0.9252887994818647
knn coefficient: 0.11781279952493488
Most trusted model: random_forest


The meta-model is Linear Regression.
It learns how much to trust each base model.
The model with the largest coefficient is trusted the most.